# Download PR review comment dataset

Pipeline: GH Archive → filter by top snapshot commits → GraphQL enrichment (`pr_title`, `pr_body`, `repo_star_count`, `is_resolved`) → unified code enrichment (compare patches, base/snapshot zipballs, patched content, dependency resolution).

Set `GITHUB_TOKEN` in `.env` (or the environment) before enrichment steps. Install the package editable: `pip install -e .` from the repo root.

In [ ]:
import asyncio
import logging

import json
import aiohttp
from dotenv import load_dotenv

from ai_code_reviewer.dataset import (
    checkpoints,
    gh_archive,
    github_api,
    github_graphql,
)
from ai_code_reviewer.dataset import config as dataset_config

load_dotenv()
logging.basicConfig(level=logging.INFO)

In [ ]:
gh_archive_semaphore = asyncio.Semaphore(dataset_config.GH_ARCHIVE_CONCURRENCY)
gh_semaphore = asyncio.Semaphore(dataset_config.GITHUB_API_CONCURRENCY)

In [ ]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GH_ARCHIVE_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    dataset = await gh_archive.fetch_pr_comments_range(
        session,
        dataset_config.RANGE_START,
        dataset_config.RANGE_END,
        gh_archive_semaphore,
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_RAW_PATH)

In [ ]:
dataset = gh_archive.filter_dataset_by_top_snapshot_commits(
    dataset, 5#dataset_config.SNAPSHOT_COMMITS_TO_KEEP
)

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FILTERED_PATH)

In [ ]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GITHUB_API_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    await github_graphql.enrich_dataset_with_graphql_info(
        dataset, session, gh_semaphore
    )

In [ ]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GITHUB_API_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    await github_api.enrich_dataset_with_code(
        dataset, session, gh_semaphore
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FINAL_PATH)

In [ ]:
dataset["tomwojcik/starlette-context"]["59"]["commits"]["37cafcf57c626da944d3c15be7e48c54919efd54"]["starlette_context/middleware/mixin.py"]["dependencies"].keys()

In [ ]:
print(dataset["tomwojcik/starlette-context"]["59"]["commits"]["37cafcf57c626da944d3c15be7e48c54919efd54"]["tests/test_plugins/test_error_responses.py"]["patched_content"])

In [ ]:
# Validate dependency content source:
#   - If a dependency file was itself changed in the same snapshot commit,
#     its content in `dependencies` is the annotated patched view (+/- markers).
#   - If a dependency file was NOT changed, its content is raw HEAD text from
#     the snapshot zipball.
patched_src_deps: list[tuple[str, str, str, str, str]] = []
zipball_src_deps: list[tuple[str, str, str, str, str]] = []

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        for commit_sha, path_map in pr_entry["commits"].items():
            for path, file_entry in path_map.items():
                deps = file_entry.get("dependencies") or {}
                for dep_path, dep_content in deps.items():
                    record = (repo_name, pr_number, commit_sha[:7], path, dep_path)
                    if dep_path in path_map:
                        patched_src_deps.append(record)
                    else:
                        zipball_src_deps.append(record)

print(f"Dependencies sourced from patched_content (dep also changed in snapshot): {len(patched_src_deps)}")
for repo, pr, sha, src, dep in patched_src_deps[:5]:
    print(f"  [{repo} PR#{pr} @{sha}]  {src}  ->  {dep}")

print()
print(f"Dependencies sourced from snapshot zipball (unchanged dep files):          {len(zipball_src_deps)}")
for repo, pr, sha, src, dep in zipball_src_deps[:5]:
    print(f"  [{repo} PR#{pr} @{sha}]  {src}  ->  {dep}")

In [ ]:
# Compute dataset statistics
num_prs = 0
num_snapshot_commits = 0
num_files_with_comments = 0
num_files_without_comments = 0
num_resolved_comments = 0
num_unresolved_comments = 0
balance_violations = 0  # snapshots where no-comment files exceed commented files
num_files_with_deps = 0
total_dep_files = 0

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        num_prs += 1
        for commit_sha, path_map in pr_entry["commits"].items():
            num_snapshot_commits += 1
            snap_with = 0
            snap_without = 0
            for path, file_entry in path_map.items():
                comments = file_entry.get("comments", [])
                for comment in comments:
                    if comment.get("is_resolved", False):
                        num_resolved_comments += 1
                    else:
                        num_unresolved_comments += 1
                if comments:
                    snap_with += 1
                else:
                    snap_without += 1
                deps = file_entry.get("dependencies", {})
                if deps:
                    num_files_with_deps += 1
                    total_dep_files += len(deps)
            num_files_with_comments += snap_with
            num_files_without_comments += snap_without
            if snap_without > snap_with:
                balance_violations += 1

num_files = num_files_with_comments + num_files_without_comments
num_comments = num_resolved_comments + num_unresolved_comments

print(f"Number of PRs:                  {num_prs}")
print(f"Number of snapshot commits:     {num_snapshot_commits}")
print(f"Number of files (total):        {num_files}")
print(f"  - with comments:              {num_files_with_comments}")
print(f"  - without comments:           {num_files_without_comments}")
print(f"Number of comments (total):     {num_comments}")
print(f"  - resolved:                   {num_resolved_comments}")
print(f"  - unresolved:                 {num_unresolved_comments}")
print(
    f"Balance violations (snapshots where no-comment files > commented files): {balance_violations}"
)
print()
dep_pct = 100 * num_files_with_deps / num_files if num_files else 0.0
avg_deps = total_dep_files / num_files_with_deps if num_files_with_deps else 0.0
print(f"Files with >=1 dependency:      {num_files_with_deps} ({dep_pct:.1f}%)")
print(f"Total dependency file entries:  {total_dep_files}")
print(f"Avg dependencies per file:      {avg_deps:.2f}")

In [ ]:
# Convert dataset to JSON with list of files
files_list = []

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        pr_title = pr_entry.get("pr_title")
        pr_body = pr_entry.get("pr_body")
        repo_star_count = pr_entry.get("repo_star_count")
        for commit_sha, path_map in pr_entry["commits"].items():
            for path, file_entry in path_map.items():
                file_obj = {
                    "repo": repo_name,
                    "pr_number": pr_number,
                    "pr_title": pr_title,
                    "pr_body": pr_body,
                    "repo_star_count": repo_star_count,
                    "commit_sha": commit_sha,
                    "path": path,
                    "patched_content": file_entry.get("patched_content"),
                    # {path: content} map of direct in-repo imported files resolved
                    # from the HEAD version of this file; absent when none resolved.
                    "dependencies": file_entry.get("dependencies"),
                    "comments": [
                        {
                            "body": comment.get("body"),
                            "is_resolved": comment.get("is_resolved"),
                            "annotated_start_line": comment.get("annotated_start_line"),
                            "annotated_end_line": comment.get("annotated_end_line"),
                        }
                        for comment in file_entry.get("comments", [])
                    ],
                }
                files_list.append(file_obj)
with open("files_list.json", "w") as f:
    json.dump(files_list, f)

In [ ]:
with open("./files_list.json", "r") as f:
    files = json.load(f)

In [ ]:
files